# CatBoost
Training CatBoost on the preprocessed and cleaned up features.

In [ ]:
from utils.preprocessing import ExtendedPreprocessPipeline, FeaturePreprocessPipeline
import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from utils.preprocessing import preprocess_currency
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
from math import sqrt

In [ ]:
pd.set_option('display.max_columns', 50)

In [ ]:
df_test = ExtendedPreprocessPipeline.run(pd.read_csv('data/sales_ads_test.csv'))
df_train = ExtendedPreprocessPipeline.run(pd.read_csv('data/sales_ads_train.csv'))
df_data = pd.concat((df_train, df_test), ignore_index=True, copy=True).reset_index()
len(df_test), len(df_train)

In [ ]:
df_data = FeaturePreprocessPipeline.run(df_data)

In [ ]:
# Too extreme outliers make less reliable validation
df_data.drop(index=df_data.index[df_data.Cena > 6_000_000], inplace=True)

In [ ]:
# Selected features
cat_features = ['Marka_pojazdu', 'Model_pojazdu', 'Generacja_pojazdu', 
                'Naped', 'Skrzynia_biegow', 'Typ_nadwozia', 'Kolor', 'Liczba_drzwi',
                'Kraj_pochodzenia', 'Miasto', 'Wojewodztwo']
binary_features = [
    'Stan_binary'
]
num_features=['Przebieg_km', 'Moc_KM', 'Pojemnosc_cm3', 'Rok_produkcji',
              'Count_wyposazenia']
features = num_features + binary_features

# Encode the targets for the
assert np.all(df_test.ID.to_numpy() == df_data.loc[df_data.Cena.isna(), 'ID'].to_numpy()), "General data frame is incorrectly merged."
# Fill the missing brands with their equivalents
print("Maybach is chainged into Rolls-Royce for testing as it does not appear in training.")
df_test.loc[df_test.Marka_pojazdu == 'Maybach', 'Marka_pojazdu'] = 'Rolls-Royce'
print("Brands, which are in test, but not train. They will be set to Nan.")
print(df_test.loc[~np.isin(df_test.Marka_pojazdu, df_train.Marka_pojazdu.unique()), 'Marka_pojazdu'].to_numpy())
df_test.loc[~np.isin(df_test.Marka_pojazdu, df_train.Marka_pojazdu.unique()), 'Marka_pojazdu'] = np.nan

df = df_data.loc[~df_data.Cena.isna()].copy(deep=True).reset_index()
df_submition = df_data.loc[df_data.Cena.isna()].copy(deep=True).reset_index()

for cat in cat_features:
    df.loc[:, cat] = df.loc[:, cat].astype('string').fillna('Other')
    df_submition.loc[:, cat] = df_submition.loc[:, cat].astype('string').fillna('Other')

## Training
Training CatBoost on K-Fold validation.

In [ ]:
##### Split the data into K folds, but preserve price distribution
num_folds = 6
target_column='Cena'
df.sort_values(by=target_column, inplace=True, kind='stable')  # Split randomly using modulo, but having even price distribution
folds_idx = np.arange(len(df))
for i in folds_idx:
    folds_idx[i] = i % num_folds
folds = []
for i in range(num_folds):
    test_mask = folds_idx == i % num_folds
    folds.append((np.argwhere(~test_mask).flatten(), np.argwhere(test_mask).flatten()))
y = df[target_column]

# XGBoost parameters
params = {
    'depth': 10,                     
    'learning_rate': 0.025,           
    'subsample': 0.8,                
    'rsm': 0.8,                      
    'min_data_in_leaf': 10,         
    'random_seed': 100,              
    'l2_leaf_reg': 10,               
    'loss_function': 'RMSE',         
    'eval_metric': 'RMSE',           
    'one_hot_max_size': 0,           # disable one-hot encoding
    'bootstrap_type': 'Bernoulli',
    'iterations': 5_000,
    'verbose': 100,
}

# Cross-validation stats
cv_results = {
    'test-rmse-mean': [],
    'test-rmse-std': [],
    'train-rmse-mean': [],
    'train-rmse-std': []
}

fold_scores = []
hardest_examples = []
models = []
scores = []
best_model = None
best_rmse = 1_000_000
safe_num_of_estimators = 0
for i, (train_idx, test_idx) in enumerate(folds):
    print(f"Fold {i+1}/{num_folds}")

    X_train, y_train = df.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = df.iloc[test_idx], y.iloc[test_idx]
    X_train, X_test = X_train[features + cat_features], X_test[features + cat_features]

    dtrain = Pool(X_train, y_train, cat_features=cat_features)
    dtest = Pool(X_test, y_test, cat_features=cat_features)

    # Train the model
    model = CatBoostRegressor(**params)
    model.fit(dtrain, eval_set=dtest, early_stopping_rounds=50)

    # Get predictions
    train_preds = model.predict(dtrain)
    test_preds = model.predict(dtest)

    # Calculate metrics
    train_rmse = sqrt(mean_squared_error(y_train, train_preds))
    test_rmse = sqrt(mean_squared_error(y_test, test_preds))
    
    # Save model if it has the lowest RMSE
    if test_rmse < best_rmse:
        best_rmse = test_rmse
        best_model = model
    safe_num_of_estimators = (safe_num_of_estimators * i + model.best_iteration_) / (i + 1)
    
    # Get the examples, which were the hardest to classify
    test_mse = (y_test - test_preds) ** 2
    hardest_args = np.argsort(test_mse)[-5:]
    hardest_validation_examples = X_test.iloc[hardest_args].copy(deep=True)
    hardest_validation_examples.loc[:, 'Prediction'] = test_preds[hardest_args]
    hardest_validation_examples.loc[:, 'Cena'] = y_test.iloc[hardest_args]
    hardest_examples.append(hardest_validation_examples)

    print(f"Fold {i+1} - Train RMSE: {train_rmse:.6f}, Test RMSE: {test_rmse:.6f}")
    fold_scores.append((train_rmse, test_rmse))
    models.append(model)
    scores.append(test_rmse)

# Stats on best model
print(f"Best validation RMSE: {best_rmse:.6f}")
print(f"Average number of estimators: {safe_num_of_estimators}")

# Summarize results
train_scores = [score[0] for score in fold_scores]
test_scores = [score[1] for score in fold_scores]

cv_results['train-rmse-mean'] = np.mean(train_scores)
cv_results['train-rmse-std'] = np.std(train_scores)
cv_results['test-rmse-mean'] = np.mean(test_scores)
cv_results['test-rmse-std'] = np.std(test_scores)

print("\nCross-Validation Results:")
print(f"Train RMSE: {cv_results['train-rmse-mean']:.6f} ± {cv_results['train-rmse-std']:.6f}")
print(f"Test RMSE: {cv_results['test-rmse-mean']:.6f} ± {cv_results['test-rmse-std']:.6f}")

In [ ]:
model.get_feature_importance(dtrain)

## Train the final model
Traininig the final model is performed on the whole dataset to maximise the gain from the data. In such case one have
to be extra careful about overfitting.

In [ ]:
# Create very small validation set for final training early stopping
modulo = 1000
df.sort_values(by=target_column, inplace=True, kind='stable')  # Split randomly using modulo, but having even price distribution

test_mask = np.arange(len(df)) % modulo == 0
train_idx, test_idx = np.argwhere(~test_mask).flatten(), np.argwhere(test_mask).flatten()

y = df[target_column]

X_train, y_train = df.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = df.iloc[test_idx], y.iloc[test_idx]
X_submition = df_submition
all_features = num_features + cat_features
X_train, X_test, X_submition = X_train.loc[:, all_features], X_test.loc[:, all_features], X_submition.loc[:, all_features]

# Create DMatrix objects for XGBoost
dtrain = Pool(X_train, y_train, cat_features=cat_features)
dtest = Pool(X_test, y_test, cat_features=cat_features)
# Train the model
params_final = copy.deepcopy(params)
params_final['iterations'] = 2_500
model = CatBoostRegressor(**params_final)
model.fit(dtrain, eval_set=dtest, early_stopping_rounds=50)

# Predict
dsub = Pool(X_submition, cat_features=cat_features)
predictions = model.predict(dsub)
print(f"Maximal prediction value detected : {predictions.max()}")
print(f"Maximal target in train : {y_train.max()}")
print(f"Maximal target in test : {y_test.max()}")

In [ ]:
model.save_model('submission_model.json')

df_submit = df_test.copy(deep=True)
df_submit['Cena'] = predictions
df_submit = preprocess_currency(df_submit, invert=True)

df_submit.drop(columns=[col for col in df_test.columns if col not in ['ID', 'Cena']], inplace=True)
display(df_submit)
df_submit.to_csv('submission.csv', index=False)

In [ ]:
df_test.iloc[np.argsort(predictions)[-2]]

In [ ]:
np.sort(predictions)[-1]

In [ ]:
df_submit = df_test.copy(deep=True)
df_submit['Cena'] = predictions
df_submit = preprocess_currency(df_submit, invert=True)

mask = df_submit.Waluta == 'EUR'
print(predictions[mask][:10], df_submit.Cena[mask].to_numpy()[:10])